In [1]:
import requests
import logging
import psycopg2
from apscheduler.schedulers.background import BlockingScheduler
from dotenv import load_dotenv
import os


### **1. Error Handling**

* Fetch data from the JSONPlaceholder API ("https://jsonplaceholder.typicode.com/end-point"). Implement robust error handling to manage scenarios where:
    * The API endpoint is invalid.
    * The API returns a non-success status code.
    * The network connection fails or times out.
    * Log all errors to a file named api_errors.log with details such as the timestamp, error type, and error message.

In [2]:
# configuring logging
filelogger = logging.getLogger(__name__)
filelogger.level = logging.INFO
handler = logging.FileHandler('api_errors.log', mode='a')
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
handler.setFormatter(formatter)
filelogger.addHandler(handler)


def fetch_api_data(endpoint):
    url = f'https://jsonplaceholder.typicode.com/{endpoint}'

    try:
        response = requests.get(url, timeout=5)

        if response.status_code != 200:
            error_msg = f'Non-success status code: {response.status_code} for endpoint {endpoint}'
            filelogger.error(f'status code error :- {error_msg}')
            print(error_msg)
            return None
        

        return response

    except requests.exceptions.Timeout as e:
        filelogger.error(f'Timeout Error :- {str(e)}')
        print('Request Timeout Error')

    except requests.exceptions.ConnectionError as e:
        filelogger.error(f'Connection Error :- {str(e)}')
        print('Network connection Failed')

    except requests.exceptions.RequestException as e:
        logging.error(f"Request Exception :- {str(e)}")
        print('General request error occurred')

    except Exception as e:
        filelogger.error(f'Unexpected Error Occurred :- {str(e)}')
        print('Unexpected Error Occurred')

    return None

### **2. Database Storage with PostgreSQL**

* Set up a PostgreSQL database named api_database with the following tables:
    * Users api endpoint: https://jsonplaceholder.typicode.com/users
    * Users table should contain following columns: id, name, email, address (formatted as a single string).
    * Posts api endpoint: https://jsonplaceholder.typicode.com/posts
    * Posts table should contain following columns: id, user_id, title, body.

* Use Python's psycopg2 or sqlalchemy library to connect to the database.
    * Fetch and store user data from the API's /users endpoint into the Users table.
    * Fetch and store post data from the API's /posts endpoint into the Posts table.
    * Ensure duplicate entries are avoided by using ON CONFLICT DO NOTHING or similar techniques.

In [6]:
user_data = fetch_api_data("users")
user_data_json = user_data.json()
# print(f'{user_data_json} \n')

post_data = fetch_api_data('posts')
post_data_json = post_data.json()
# print(f'{post_data_json} \n')

In [4]:
load_dotenv()

conn = psycopg2.connect(
    host=os.getenv('db_host'),
    database=os.getenv('db_database'),
    user=os.getenv('db_user'),
    password=os.getenv('db_password')
)

cursor = conn.cursor()

for user in user_data_json:
    address_str = f"{user['address']['street']}, {user['address']['suite']}, {user['address']['city']}, {user['address']['zipcode']} ({user['address']['geo']['lat']}, {user['address']['geo']['lng']})"
    cursor.execute(
        '''
        INSERT INTO api_schema.users(id, name, email, address)
        VALUES (%s, %s, %s, %s)
        ON CONFLICT (id) DO NOTHING
        ''',
        (
            user['id'],
            user['name'], 
            user['email'],
            address_str
        )
    )

for post in post_data_json:
    cursor.execute(
        '''
        INSERT INTO api_schema.posts(id, user_id, title, body)
        VALUES (%s, %s, %s, %s)
        ON CONFLICT (id) DO NOTHING
        ''',
        (
            post['id'],
            post['userId'], 
            post['title'],
            post['body']
        )
    )

conn.commit()

cursor.close()
conn.close()

print("Data inserted successfully")

Data inserted successfully


### **3. Scheduled API Calls**

* Use the schedule or APScheduler library to run a task every 10 minutes. The task should:
    * Fetch new posts from the /posts endpoint.
    * Insert only new posts into the database.
    * Log each execution with the timestamp and the number of new posts added to the database.
    * If an error occurs during the scheduled task (e.g., the API is unavailable), log the error to api_errors.log.


### **4. Summary Report**

* After each scheduled task execution, print a summary report showing:
    * The total number of users in the database.
    * The total number of posts in the database.
    * The number of new posts added during the most recent execution.
    * Ensure the script runs indefinitely until manually stopped.

In [5]:
def fetch_data():
    try: 
        new_post_data = fetch_api_data('posts')
        post_json_data = new_post_data.json()

        load_dotenv()

        conn = psycopg2.connect(
            host = os.getenv('db_host'),
            database=os.getenv('db_database'),
            user=os.getenv('db_user'),
            password=os.getenv('db_password') 
        )

        cursor = conn.cursor()

        new_posts = 0

        for post in post_json_data:
            cursor.execute(
                '''
                INSERT INTO api_schema.posts(id, user_id, title, body)
                VALUES (%s, %s, %s, %s)
                ON CONFLICT (id) DO NOTHING
                ''',
                (
                    post['id'],
                    post['userId'], 
                    post['title'],
                    post['body']
                )
            )

            if cursor.rowcount == 1:
                new_posts += 1

        conn.commit()

        cursor.execute("SELECT COUNT(*) FROM api_schema.users")
        total_users = cursor.fetchone()[0]

        cursor.execute("SELECT COUNT(*) FROM api_schema.posts")
        total_posts = cursor.fetchone()[0]

        print('---------- Summary Report ----------')
        print("Total Users: ", total_users)
        print("Total Posts: ", total_posts)
        print("New posts added during Execution: ", new_posts)
        print('\n')

        cursor.close()
        conn.close()
        
        filelogger.info(f"{new_posts} new posts inserted")

    except Exception as e:
        filelogger.error(f"Scheduled task failed: {str(e)}")

scheduler = BlockingScheduler()
scheduler.add_job(fetch_data, 'interval', seconds = 5)

scheduler.start()

---------- Summary Report ----------
Total Users:  10
Total Posts:  100
New posts added during Execution:  0


---------- Summary Report ----------
Total Users:  10
Total Posts:  100
New posts added during Execution:  0


---------- Summary Report ----------
Total Users:  10
Total Posts:  100
New posts added during Execution:  0


---------- Summary Report ----------
Total Users:  10
Total Posts:  100
New posts added during Execution:  0


---------- Summary Report ----------
Total Users:  10
Total Posts:  100
New posts added during Execution:  0




KeyboardInterrupt: 